# 05. Classification by Reconstruction — Visual Walkthrough

**Illustrative only.** This notebook visualises *how* the generative classifier
("Classification by Reconstruction", CbR) reaches a decision, so a reviewer can
understand the mechanism. The quantitative evaluation (accuracy, F1, slide-level
bootstrap CIs) is done in **notebook 09** — nothing here computes a reported metric.

### The idea
The CVAE (notebook 03) is a *conditional* generator: given a latent code `z` and a
class label, its decoder reconstructs a spectrum *as if* it belonged to that class.
To classify a real spectrum `x`, we ask for each candidate class `k`:

> *"If `x` were class `k`, what would it look like — and how close is that to the real `x`?"*

1. **Encode** `x → z` with the fold's CVAE encoder (we use the latent mean `z_mean`).
2. **Decode** `(z, k)` for every class `k` → a hypothetical reconstruction `x̂ₖ`.
3. **Score** each with reconstruction MSE: `MSEₖ = mean((x − x̂ₖ)²)`.
4. **Predict** the class with the *lowest* MSE — the hypothesis that best explains `x`.

This is exactly the rule scored in notebook 09 / `pipeline.evaluation_stage`
(`_cbr_predictions`). Here we reuse the same model-loading helper
(`pipeline._cvae_gen.load_encoder_decoder`) and the same MSE definition, so the
picture matches the numbers.

**Inputs:** a fold's CVAE weights under `models_cvae_dir` — run notebook 03 first.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebooks.experiment_utils import setup_experiment
from pipeline._common import make_loader, num_classes_of
from pipeline._cvae_gen import build_cvae_config, load_encoder_decoder

# ── Selection ──────────────────────────────────────────────────────────
CONFIG_NAME = 'exp_201'    # an experiment whose CVAE weights exist (run nb 03 first)
FOLD_NAME   = 'fold_0'

config, _ = setup_experiment(CONFIG_NAME)
num_classes = num_classes_of(config)
CLASS_NAMES = (['Mutant', 'Wildtype'] if num_classes == 2
               else [f'class {i}' for i in range(num_classes)])

# Real validation spectra for the chosen fold (same binary mapping as training)
loader = make_loader(config)
n_features = loader.X.shape[1]
fold = next(fd for fd in loader if fd['fold_name'] == FOLD_NAME)
X_val = fold['val'][0].astype(np.float32)
y_val = fold['val'][1].copy()
if y_val.min() == 1:
    y_val = y_val - 1
if num_classes == 2:
    y_val = np.where(y_val < 3, 0, 1)

# Load this fold's trained CVAE (same helper the evaluation stage uses — no drift)
cvae_config = build_cvae_config(config, n_features)
encoder, decoder = load_encoder_decoder(
    cvae_config, config['paths']['models_cvae_dir'], FOLD_NAME)

print(f"{CONFIG_NAME} {FOLD_NAME}: {X_val.shape[0]} validation spectra, "
      f"{n_features} bands, {num_classes} classes {CLASS_NAMES}")

In [ ]:
def cbr_reconstructions(X):
    """Encode X, then decode under every class hypothesis.

    Returns:
      recons[k] : (N, n_features) reconstruction of X assuming class k
      mse[:, k] : per-spectrum reconstruction MSE for class k (the notebook-09 score)
    """
    z_mean = encoder.predict(X.reshape(-1, n_features, 1), batch_size=256, verbose=0)[0]
    recons, mse = [], np.zeros((len(X), num_classes), np.float32)
    for k in range(num_classes):
        label = tf.keras.utils.to_categorical([k] * len(X), num_classes=num_classes).astype('float32')
        rec = decoder.predict([z_mean, label], batch_size=256, verbose=0).reshape(len(X), -1)
        recons.append(rec)
        mse[:, k] = np.mean((X - rec) ** 2, axis=1)
    return recons, mse

## 1. One spectrum, every class hypothesis

Reconstruct a single real spectrum under each candidate class and overlay the results. The reconstruction that hugs the real spectrum most tightly (lowest MSE) is the prediction.

In [ ]:
ex = 0   # index into X_val — try others to see different cases
recons, mse = cbr_reconstructions(X_val[ex:ex + 1])
pred = int(np.argmin(mse[0]))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(X_val[ex], 'k', lw=2.0, label='real spectrum')
for k in range(num_classes):
    ax.plot(recons[k][0], '--', alpha=0.85,
            label=f'recon as {CLASS_NAMES[k]} (MSE={mse[0, k]:.4f})')
ok = 'correct' if pred == y_val[ex] else 'WRONG'
ax.set_title(f"True = {CLASS_NAMES[y_val[ex]]}   →   CbR predicts {CLASS_NAMES[pred]}  ({ok})")
ax.set_xlabel('wavenumber index'); ax.set_ylabel('intensity')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. The decision matrix over a sample

For a small balanced sample of validation spectra, the per-class reconstruction MSE. The lowest-MSE cell in each row (red ring) is the CbR prediction; when it lands on the row's true class, CbR is correct for that spectrum.

In [ ]:
rng = np.random.default_rng(0)
sample_idx = []
for k in range(num_classes):
    idx_k = np.where(y_val == k)[0]
    sample_idx.extend(rng.choice(idx_k, size=min(5, len(idx_k)), replace=False))
sample_idx = np.array(sample_idx)

_, mse_s = cbr_reconstructions(X_val[sample_idx])
pred_s = np.argmin(mse_s, axis=1)

fig, ax = plt.subplots(figsize=(1.6 * num_classes + 3, 0.45 * len(sample_idx) + 1.5))
im = ax.imshow(mse_s, aspect='auto', cmap='viridis')
ax.set_xticks(range(num_classes)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(len(sample_idx)))
ax.set_yticklabels([f"true={CLASS_NAMES[y_val[i]]}" for i in sample_idx], fontsize=8)
for r, p in enumerate(pred_s):
    ax.scatter(p, r, marker='o', s=70, facecolors='none', edgecolors='red', linewidths=1.6)
ax.set_xlabel('class hypothesis')
ax.set_title('Reconstruction MSE per class\n(red ring = lowest MSE = CbR prediction)')
fig.colorbar(im, ax=ax, label='MSE')
plt.tight_layout(); plt.show()